# 잠긴 종 · 시나리오 RAG

현재 방, 대화 상대, 플레이어에게 공개된 기록으로 NPC가 참고할 근거를 찾습니다. 같은 검색 코드에 다른 대화 상대와 공개 기록을 넘길 수 있습니다. 기본 실행에는 실제 엔진에서 수집한 다섯 NPC의 공개 전후 기록이 들어 있습니다.

위에서부터 실행하면 검색 준비, 자료 구조, 근거 선택, 결과 비교를 순서대로 볼 수 있습니다. 아래 출력은 이 PC에서 실행해 저장했습니다. Qwen은 웹 데모에서 찾은 근거를 받아 대사를 씁니다.

## 1. 실행 환경 확인

이 키워드 검색 경로는 Python 기본 라이브러리로 동작합니다.

In [ ]:
import platform
import sys

print("Python:", sys.version.split()[0])
print("운영체제:", platform.system())
print("실행 방식: CPU 검색")

## 2. 검색어와 색인 준비

질문과 공개된 문장을 같은 방식으로 잘라 비교합니다. 어떤 표현이 서로 겹치는지로 순위를 정합니다.

In [ ]:
from collections import Counter
from pathlib import Path
from uuid import uuid4
import json
import math
import re

def search_terms(text):
    # 한국어는 글자 조각, 영어는 단어로 나눈다.
    words = re.findall(r"[가-힣]+|[a-z0-9]+", text.lower())
    terms = []
    for word in words:
        if re.fullmatch(r"[가-힣]+", word):
            terms.extend(word[i:i + 2] for i in range(len(word) - 1))
        elif len(word) > 2 or word.isdigit() or word in ("ac", "dc", "hp"):
            terms.append(word[:-1] if len(word) > 3 and word.endswith("s") else word)
    return terms

class LexicalIndex:
    def __init__(self):
        self.rows = {}

    def add(self, ids, documents, metadatas):
        for key, text, meta in zip(ids, documents, metadatas):
            terms = Counter(search_terms(f"{meta.get('title', '')} {text}"))
            self.rows[key] = (text, meta, terms, sum(terms.values()))

    def search(self, question, limit, where):
        terms = set(search_terms(question))
        allowed = None if where is None else set(where["card_id"]["$in"])
        rows = [row for row in self.rows.values() if allowed is None or row[1]["card_id"] in allowed]
        if not terms or not rows:
            return []
        avg = max(1, sum(row[3] for row in rows) / len(rows))
        idf = {}
        for term in terms:
            freq = sum(term in row[2] for row in rows)
            idf[term] = math.log1p((len(rows) - freq + 0.5) / (freq + 0.5))

        # 흔하지 않은 검색어가 겹칠수록 점수를 높인다.
        def rank(row):
            _, _, counts, length = row
            return sum(idf[t] * counts[t] / (counts[t] + 0.3 + 0.9 * length / avg) for t in terms if counts[t])

        return [(text, meta) for text, meta, _, _ in sorted(rows, key=rank, reverse=True)[:limit]]

## 3. 공개 카드 조각을 검색 기능에 연결

카드 문장을 검색하기 좋은 조각으로 나눕니다. 원래 카드 ID를 보관해 나중에 허용된 카드만 고를 수 있게 합니다.

In [ ]:
# 웹 데모의 CPU 키워드 경로를 실행한다.
embedder = None
client = None

def create_collection(name):
    if embedder is None:
        return LexicalIndex()
    return client.get_or_create_collection(
        name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
    )

def add_documents(collection, ids, documents, metadatas):
    if isinstance(collection, LexicalIndex):
        collection.add(ids, documents, metadatas)
    else:
        vectors = embedder.encode([f"passage: {text}" for text in documents])
        collection.upsert(ids=ids, documents=documents, embeddings=vectors.tolist(), metadatas=metadatas)

def query(collection, question, limit=3, where=None):
    if isinstance(collection, LexicalIndex):
        return collection.search(question, limit, where)
    count = collection.count()
    if count == 0:
        return []
    vector, = embedder.encode([f"query: {question}"])
    result = collection.query(query_embeddings=[vector.tolist()], n_results=min(count, limit), where=where)
    return list(zip(result["documents"][0], result["metadatas"][0]))

def split_scenario(text):
    body = re.sub(r"^\[[^\]]+\]\s*", "", text)
    parts = [part.strip() for part in re.split(r"(?<=[.!?\"”])\s+(?=[가-힣A-Z\"“])", body) if part.strip()]
    return [
        " ".join(parts[i:i + 2]) if i + 1 < len(parts) and len(part) + len(parts[i + 1]) <= 420 else part
        for i, part in enumerate(parts)
    ]

def npc_name(card):
    if card["kind"] != "npc":
        return None
    name = card.get("speaker")
    if name:
        return name
    # 인물 카드의 '이름 — 소개' 머리말을 사용합니다.
    found = re.match(r"^([가-힣]{2,12})\s*[—–-]\s+", card["player_text"])
    return found[1] if found else None

## 4. 플레이 데이터 가져오기

여기에는 실제로 플레이어에게 보인 문장과 카드의 방·유형·NPC 이름만 들어 있습니다. 자기 기록을 읽는 코드는 9번에 있습니다.

In [ ]:
# 실제 게임에서 수집한 공개 기록. 아래 NPC 이름은 검색 분기에 쓰지 않는다.
DEMO_CARDS = {'scn-g-halvik-house': {'id': 'scn-g-halvik-house', 'room': 'halvik_house', 'kind': 'desc'},
 'scn-g-well-night': {'id': 'scn-g-well-night', 'room': 'well_side', 'kind': 'desc'},
 'scn-m-market-desc': {'id': 'scn-m-market-desc', 'room': 'village_market', 'kind': 'desc'},
 'scn-m-npc-innkeeper': {'id': 'scn-m-npc-innkeeper', 'room': 'village_market', 'kind': 'npc', 'speaker': '오단'},
 'scn-m-npc-smith': {'id': 'scn-m-npc-smith', 'room': 'village_market', 'kind': 'npc', 'speaker': '그란'},
 'scn-q1-npc-sera': {'id': 'scn-q1-npc-sera', 'room': 'village_square', 'kind': 'npc', 'speaker': '세라'},
 'scn-q4-npc-mera-song': {'id': 'scn-q4-npc-mera-song',
                          'room': 'well_side',
                          'kind': 'npc',
                          'known_by_npcs': ['메라'],
                          'player_text': ''},
 'scn-s1-desc': {'id': 'scn-s1-desc', 'room': 'village_square', 'kind': 'desc'},
 'scn-s1-npc-halvik': {'id': 'scn-s1-npc-halvik', 'room': 'halvik_house', 'kind': 'npc', 'speaker': '할빅'},
 'scn-s1-npc-mera': {'id': 'scn-s1-npc-mera', 'room': 'well_side', 'kind': 'npc', 'speaker': '메라'}}

DEMO_CASES = [{'room': 'village_square',
  'speaker_id': 'scn-q1-npc-sera',
  'speaker': '세라',
  'before': {'scn-s1-desc': ('scn-s1-desc',
                             '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. '
                             '마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 '
                             '희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.')},
  'after': {'scn-s1-desc': ('scn-s1-desc',
                            '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 '
                            '한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 '
                            '울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
            'scn-q1-npc-sera': ('scn-q1-npc-sera',
                                '[인물] 세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다. 일행에게 매달리지 않고 낮은 목소리로 '
                                '부탁한다. "찾을 수 없으면 그렇다고만 말해 줘요." 딸이 밤마다 들고 다니던 인형을 건네며 "이걸 보면 그 애가 알아볼 거예요"라고 '
                                '한다. 남편과 딸이 사라진 날 모두 우물 쪽에서 종소리가 유난히 컸다고 기억한다.')}},
 {'room': 'halvik_house',
  'speaker_id': 'scn-s1-npc-halvik',
  'speaker': '할빅',
  'before': {'scn-s1-desc': ('scn-s1-desc',
                             '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. '
                             '마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 '
                             '희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
             'scn-g-halvik-house': ('scn-g-halvik-house',
                                    '[장면 묘사] 촌장 집. 채석장이 한창일 때 지은 큰 집이라 방이 넓고 허름하다. 벽난로 위에 낡은 석판이 걸려 있고, 안쪽 서재로 '
                                    '가는 문이 반쯤 열려 있다.')},
  'after': {'scn-s1-desc': ('scn-s1-desc',
                            '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 '
                            '한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 '
                            '울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
            'scn-g-halvik-house': ('scn-g-halvik-house',
                                   '[장면 묘사] 촌장 집. 채석장이 한창일 때 지은 큰 집이라 방이 넓고 허름하다. 벽난로 위에 낡은 석판이 걸려 있고, 안쪽 서재로 '
                                   '가는 문이 반쯤 열려 있다.'),
            'scn-s1-npc-halvik': ('scn-s1-npc-halvik',
                                  '[인물] 할빅 — 에코 타운의 촌장. 60대 남성, 전직 채석공. 어깨가 넓고 목소리가 크다. 일행을 의뢰인으로 맞이하며 선금 50 '
                                  'gp를 내민다. "우물을 되살려 주시오. 물만 돌아오면 되오." "60피트쯤 내려가면 물이 새는 틈이 있소. 거기만 막아 주면 되오. '
                                  '그 아래는 물에 잠겨 갈 수도 없고, 갈 까닭도 없소."')}},
 {'room': 'well_side',
  'speaker_id': 'scn-s1-npc-mera',
  'speaker': '메라',
  'before': {'scn-s1-desc': ('scn-s1-desc',
                             '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. '
                             '마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 '
                             '희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
             'scn-g-well-night': ('scn-g-well-night',
                                  '[장면 묘사] 우물가의 밤. 해가 지면 땅 밑에서 종소리가 희미하게 올라온다. 소리는 일정하지 않고 가끔 오래 멎는다. 마을 개들은 이 '
                                  '시간에 우물 근처로 오지 않는다. 우물 뚜껑 둘레의 풀은 밤이슬에도 젖지 않는다.')},
  'after': {'scn-s1-desc': ('scn-s1-desc',
                            '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 '
                            '한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 '
                            '울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
            'scn-g-well-night': ('scn-g-well-night',
                                 '[장면 묘사] 우물가의 밤. 해가 지면 땅 밑에서 종소리가 희미하게 올라온다. 소리는 일정하지 않고 가끔 오래 멎는다. 마을 개들은 이 '
                                 '시간에 우물 근처로 오지 않는다. 우물 뚜껑 둘레의 풀은 밤이슬에도 젖지 않는다.'),
            'scn-s1-npc-mera': ('scn-s1-npc-mera',
                                '[인물] 메라 — 우물지기였던 노파. 80대, 눈이 거의 보이지 않는다. 마을에서 유일하게 옛 수도원 이야기를 기억한다. "저 밑에 '
                                '수도원이 있었어. 할아버지의 할아버지 때 얘기지. 종이 울리면 아무도 우물에 가지 않았다고 했어. 울리는 게 정상이라고." 마지막 문장을 '
                                '본인도 이해하지 못한 채 되풀이한다.'),
            'scn-q4-npc-mera-song': ('scn-q4-npc-mera-song',
                                     '[인물] 메라는 물어보지 않아도 우물 곁에 앉아 입속으로 노래를 흥얼거린다. "어릴 적 할머니가 불러 준 거라우. 뜻은 모르지. '
                                     '가락만 기억해."')}},
 {'room': 'village_market',
  'speaker_id': 'scn-m-npc-innkeeper',
  'speaker': '오단',
  'before': {'scn-s1-desc': ('scn-s1-desc',
                             '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. '
                             '마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 '
                             '희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
             'scn-m-market-desc': ('scn-m-market-desc',
                                   '[장면 묘사] 대장간과 여관. 우물 광장 동쪽에 붙어 있는 두 건물이다. 마을이 작아서 재고가 적고, 외지인에게는 값을 조금 더 '
                                   '부른다는 소문이 있지만 값은 정직하다.')},
  'after': {'scn-s1-desc': ('scn-s1-desc',
                            '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 '
                            '한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 '
                            '울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
            'scn-m-market-desc': ('scn-m-market-desc',
                                  '[장면 묘사] 대장간과 여관. 우물 광장 동쪽에 붙어 있는 두 건물이다. 마을이 작아서 재고가 적고, 외지인에게는 값을 조금 더 '
                                  '부른다는 소문이 있지만 값은 정직하다.'),
            'scn-m-npc-innkeeper': ('scn-m-npc-innkeeper',
                                    '[인물] 오단 — 여관 주인. 밤마다 우물 쪽을 바라보며 장사를 한다. 여관은 안전해서 긴 휴식(8시간)을 할 수 있지만, 마을에서 '
                                    '하룻밤을 보내면 던전 밖의 시간도 흘러간다 — 종소리는 계속되고 실종은 멎지 않는다.')}},
 {'room': 'village_market',
  'speaker_id': 'scn-m-npc-smith',
  'speaker': '그란',
  'before': {'scn-s1-desc': ('scn-s1-desc',
                             '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. '
                             '마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 '
                             '희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
             'scn-m-market-desc': ('scn-m-market-desc',
                                   '[장면 묘사] 대장간과 여관. 우물 광장 동쪽에 붙어 있는 두 건물이다. 마을이 작아서 재고가 적고, 외지인에게는 값을 조금 더 '
                                   '부른다는 소문이 있지만 값은 정직하다.')},
  'after': {'scn-s1-desc': ('scn-s1-desc',
                            '[장면 묘사] 에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 '
                            '한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 '
                            '울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.'),
            'scn-m-market-desc': ('scn-m-market-desc',
                                  '[장면 묘사] 대장간과 여관. 우물 광장 동쪽에 붙어 있는 두 건물이다. 마을이 작아서 재고가 적고, 외지인에게는 값을 조금 더 '
                                  '부른다는 소문이 있지만 값은 정직하다.'),
            'scn-m-npc-smith': ('scn-m-npc-smith',
                                '[인물] 그란 — 대장장이. 채석장 연장을 손보던 사람이라 쇠붙이에는 정직하다. 쇠붙이는 원가로 팔고, 약품은 가진 만큼만 판다. 우물 '
                                '아래 이야기는 하지 않으려 한다.')}}]

print('준비된 NPC 대화 기록:',len(DEMO_CASES))

## 5. 카드와 공개 기록의 구조 확인

카드 ID는 인물이나 단서를 가리키고, 방 ID는 위치를 나타냅니다. 보조 카드의 `known_by_npcs`에는 그 정보를 아는 NPC가 적힙니다.

In [ ]:
first = DEMO_CASES[0]
print("카드 속성:", sorted(DEMO_CARDS[first["speaker_id"]]))
print("플레이 기록 항목:", sorted(first))
print("공개 조각 형식: 조각 ID → (카드 ID, 실제로 공개된 문장)")
print("대화 기록에 들어 있는 인물:", ", ".join(case["speaker"] for case in DEMO_CASES))

## 6. 현재 NPC가 아는 근거로 좁히기

현재 방에서 공개된 카드 가운데 대화 중인 NPC의 본인 카드와 그 NPC가 아는 보조 카드를 고릅니다. 검색어는 그 범위 안에서만 비교합니다.

In [ ]:
class ScenarioRAG:
    def __init__(self, cards, room, revealed_fragments):
        # 카드 원본은 분류에만 쓰고, 엔진이 실제 공개한 문장만 색인한다.
        self.cards = cards
        self.room = room
        self.public_cards = revealed_fragments
        self.indexed_cards = set()
        self.scenario_passages = {}
        self.scenario_db = None

    def search(self, question, speaker_id=None, previous_source=None):
        if speaker_id is not None:
            name = npc_name(self.cards[speaker_id])
            if not name:
                raise ValueError("대화 대상 카드에 NPC 이름이 없습니다.")
        else:
            name = None
        return self.scenario_evidence(question, speaker_id, name, previous_source)

    def search_public(self, question, card_ids=None, limit=4):
        if not self.public_cards or card_ids == []:
            return []
        if self.scenario_db is None:
            self.scenario_db = create_collection(f"drowned_bell_public_{uuid4().hex}")
        new_ids = [key for key in self.public_cards if key not in self.indexed_cards]
        if new_ids:
            passages = []
            for fragment in new_ids:
                card_id, text = self.public_cards[fragment]
                for number, passage in enumerate(split_scenario(text)):
                    passages.append((f"{fragment}:{number}", card_id, passage))
            if passages:
                documents = [text for _, _, text in passages]
                add_documents(
                    self.scenario_db, [key for key, _, _ in passages], documents,
                    [{"card_id": card_id, "passage_id": key} for key, card_id, _ in passages],
                )
                self.scenario_passages.update({key: (card_id, text) for key, card_id, text in passages})
            self.indexed_cards.update(new_ids)
        where = {"card_id": {"$in": card_ids}} if card_ids is not None else None
        return query(self.scenario_db, question, limit=limit, where=where)

    def scenario_evidence(self, question, speaker_id=None, name=None, previous_source=None):
        room = self.room
        cards = self.cards
        allowed = set()
        for card_id, _ in self.public_cards.values():
            card = cards[card_id]
            if card["room"] != room:
                continue
            if speaker_id is None:
                if card["kind"] == "desc":
                    allowed.add(card_id)
                continue
            if card_id == speaker_id or name in card.get("known_by_npcs", ()):
                allowed.add(card_id)
        # 현재 NPC의 허용 카드만 순위 계산에 넣는다.
        hits = self.search_public(question, sorted(allowed), limit=4)
        evidence = {meta["passage_id"]: text for text, meta in hits}
        selected = {}
        # 직전 근거도 현재 허용 범위에 남아 있을 때만 이어 쓴다.
        previous = self.scenario_passages.get(previous_source)
        if previous and previous[0] in allowed:
            selected[previous_source] = previous[1]
        identity = f"{speaker_id}:0"
        if speaker_id in allowed and identity in self.scenario_passages:
            selected[identity] = self.scenario_passages[identity][1]
        selected.update(evidence)
        if speaker_id is None:
            return list(selected.items())[:4]

        # Auxiliary cards often name the subject in the first sentence and give
        # the actual clue later. Restore only the fragment the player was shown.
        expanded, seen = [], set()
        for key, passage in selected.items():
            fragment = key.rsplit(":", 1)[0]
            card_id, shown = self.public_cards[fragment]
            if card_id != speaker_id and len(shown) <= 510:
                if fragment in seen:
                    continue
                seen.add(fragment)
                passage = shown
            expanded.append((key, passage))
        return expanded[:4]

## 7. 같은 질문을 여러 NPC에게 보내기

대화 상대와 공개 기록만 바꾸고 검색 함수를 그대로 실행합니다. 첫 출처와 그 구절을 비교합니다.

In [ ]:
QUESTION = "이곳이나 주변에서 아는 것이 있나요?"

for case in DEMO_CASES:
    searcher = ScenarioRAG(DEMO_CARDS, case["room"], case["after"])
    hits = searcher.search(QUESTION, case["speaker_id"])
    first_id, first_text = hits[0] if hits else ("-", "공개된 근거 없음")
    print(f"{case['speaker']} · {len(hits)}개 · {first_id}")
    print(first_text[:120], end="\n\n")

## 8. 대화 전후 공개 조건 검증

같은 질문을 대화 전과 후에 각각 넣습니다. 공개 전에는 결과가 비어 있고, 공개 후에는 대화 상대의 카드가 들어오는지 확인합니다.

In [ ]:
passed = 0
for case in DEMO_CASES:
    before = ScenarioRAG(DEMO_CARDS, case["room"], case["before"]).search(QUESTION, case["speaker_id"])
    after = ScenarioRAG(DEMO_CARDS, case["room"], case["after"]).search(QUESTION, case["speaker_id"])
    own_card = any(source_id.rsplit(":", 1)[0] == case["speaker_id"] for source_id, _ in after)
    good = not before and own_card
    print(f"{'통과' if good else '확인 필요'} · {case['speaker']} · 공개 전 {len(before)} / 공개 후 {len(after)}")
    passed += good
print(f"공개 조건 검증: {passed}/{len(DEMO_CASES)}")

## 9. 내 플레이 기록으로 바꿔 실행

이 셀은 카드 원본과 별도 플레이 기록이 있을 때 씁니다. 기록에 `room`, `speaker_id`, `revealed_fragments`를 지정합니다. 공개 조각마다 카드 ID와 실제 보여 준 문장이 필요합니다. 웹 데모에서는 `game.engine.c.cards`, `game.engine.s.room`, `game.public_cards`에서 얻습니다.

In [ ]:
CARDS_PATH = Path("cards.json")
SNAPSHOT_PATH = Path("public_snapshot.json")
# 위 검색 셀의 QUESTION을 그대로 사용한다.

if CARDS_PATH.is_file() and SNAPSHOT_PATH.is_file() and QUESTION.strip():
    raw_cards = json.loads(CARDS_PATH.read_text(encoding="utf-8"))
    cards = {card["id"]: card for card in raw_cards["cards"]}
    snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))
    search = ScenarioRAG(cards, snapshot["room"], snapshot["revealed_fragments"])
    for source_id, excerpt in search.search(
        QUESTION, snapshot.get("speaker_id"), snapshot.get("previous_source")
    ):
        print(source_id, excerpt, sep=" · ")
else:
    print("카드 원본, 현재 플레이의 공개 기록과 질문을 지정하세요.")